In [ ]:
EDVR – Free Colab GPU reconstruction
This notebook runs Nerfstudio Splatfacto for one EDVR job.

Free-tier constraint: Start and run this notebook yourself in the Colab UI. The free Colab runtime is not a persistent unattended GPU worker. Swagger creates/monitors the job; this notebook performs GPU training and automatically downloads the dataset and uploads the resulting PLY.

Before running:
1. Expose your local FastAPI backend using a temporary HTTPS tunnel (for example, Cloudflare Tunnel).
2. Set EDVR_WORKER_KEY in the backend environment and restart FastAPI.
3. In Swagger, call POST /api/reconstruction/start/{room_id} and copy its job_id.
4. In Colab, select a GPU runtime (T4 if offered).


1. Install dependencies
Cell 2 of 15

1. Install dependencies


Code cell 3
Cell 3 of 15

!nvidia-smi
!python -m pip install --upgrade pip
!python -m pip install nerfstudio requests


2. Enter the Swagger job and API connection
Cell 4 of 15

2. Enter the Swagger job and API connection


Code cell 5
Cell 5 of 15

import os, requests, zipfile, pathlib, subprocess, shlex, json, time
from getpass import getpass

API_BASE = input("Public HTTPS base URL for your FastAPI backend (no trailing slash): ").strip().rstrip("/")
JOB_ID = input("job_id copied from Swagger: ").strip()
WORKER_KEY = getpass("EDVR_WORKER_KEY: ").strip()

assert API_BASE.startswith("https://"), "Use a public HTTPS URL, not localhost."
assert JOB_ID, "job_id is required."
HEADERS = {"X-Worker-Key": WORKER_KEY}

status_url = f"{API_BASE}/api/reconstruction/status/{JOB_ID}"
r = requests.get(status_url, timeout=30)
r.raise_for_status()
job = r.json()
ROOM_ID = job["room_id"]
print("Room:", ROOM_ID, "| photos:", job.get("image_count"), "| status:", job["status"])


3. Mark job as processing and download the dataset
Cell 6 of 15

3. Mark job as processing and download the dataset


Code cell 7
Cell 7 of 15

r = requests.post(
    f"{API_BASE}/api/reconstruction/worker/start/{JOB_ID}",
    headers=HEADERS, timeout=30
)
r.raise_for_status()
print("Job status:", r.json().get("status"))

dataset_url = f"{API_BASE}/api/reconstruction/dataset/{ROOM_ID}"
r = requests.get(dataset_url, timeout=180)
r.raise_for_status()

work = pathlib.Path("/content/edvr_job")
images_dir = work / "images"
work.mkdir(parents=True, exist_ok=True)
zip_path = work / "dataset.zip"
zip_path.write_bytes(r.content)

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(work)

assert images_dir.exists() and any(images_dir.iterdir()), "No images found in downloaded dataset."
print("Downloaded", len(list(images_dir.iterdir())), "images")


4. Check GPU and process camera poses with COLMAP
Cell 8 of 15

4. Check GPU and process camera poses with COLMAP


Code cell 9
Cell 9 of 15

gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("No NVIDIA GPU is attached to this Colab runtime. Select Runtime > Change runtime type > GPU.")

data_dir = work / "processed"
cmd = [
    "ns-process-data", "images",
    "--data", str(images_dir),
    "--output-dir", str(data_dir),
    "--matching-method", "exhaustive"
]
print("Running:", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)


5. Train Splatfacto
Cell 10 of 15

5. Train Splatfacto


Code cell 11
Cell 11 of 15

train_dir = work / "outputs"
cmd = [
    "ns-train", "splatfacto",
    "--data", str(data_dir),
    "--output-dir", str(train_dir),
    "--max-num-iterations", "15000",
    "--viewer.quit-on-train-completion", "True"
]
print("Running:", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)


6. Locate the trained config and export Gaussian PLY
Cell 12 of 15

6. Locate the trained config and export Gaussian PLY


Code cell 13
Cell 13 of 15

configs = list(train_dir.glob("**/config.yml"))
if not configs:
    raise FileNotFoundError("Nerfstudio config.yml not found. Training may not have completed.")
config_path = max(configs, key=lambda p: p.stat().st_mtime)
export_dir = work / "export"
export_dir.mkdir(exist_ok=True)

cmd = [
    "ns-export", "gaussian-splat",
    "--load-config", str(config_path),
    "--output-dir", str(export_dir)
]
print("Using config:", config_path)
subprocess.run(cmd, check=True)

ply_files = list(export_dir.glob("*.ply")) + list(export_dir.glob("**/*.ply"))
if not ply_files:
    raise FileNotFoundError("No PLY was exported.")
ply_path = max(ply_files, key=lambda p: p.stat().st_size)
print("Exported:", ply_path, "| bytes:", ply_path.stat().st_size)


7. Upload the result to EDVR and verify Swagger status
Cell 14 of 15

7. Upload the result to EDVR and verify Swagger status


Code cell 15
Cell 15 of 15

with open(ply_path, "rb") as f:
    r = requests.post(
        f"{API_BASE}/api/reconstruction/result/{JOB_ID}",
        headers=HEADERS,
        files={"model": ("room_gaussian.ply", f, "application/octet-stream")},
        timeout=900
    )
r.raise_for_status()
print(json.dumps(r.json(), indent=2))

final = requests.get(status_url, timeout=30)
final.raise_for_status()
print("Final job status:", final.json().get("status"))
print("Model endpoint:", API_BASE + final.json().get("model_url", ""))